# FQ-SSIUU Mother notebook

One versioned recipe per run. The first recipe validates the complete non-model integration suite. Accepted method adapters reuse this notebook for corrected pilots; confirmatory runs require the approved frozen protocol.

Attach the private source dataset produced by `scripts/package_kaggle_source.py`. Stage this notebook with `scripts/stage_mother.py`; it supplies immutable source and orchestration identities. Select the pinned Kaggle image and one T4 (`CUDA_VISIBLE_DEVICES=0`). Credentials belong in Kaggle Secrets, never notebook cells. Model/data access is requested only by a recipe that needs it.

Every invocation gets a new run namespace. Outputs include the exact recipe, source linkage, environment/overlay lock, command stdout/stderr, termination records, and a sealed artifact manifest. Private Kaggle and Hugging Face archive receipts are required before the run is marked durably stored. `archive_run.py` can mirror using authorised credentials on the host or in the runtime; notebook outputs remain the recovery copy if synchronization fails.

## Credentials

Use the ignored local `.env` when running from your computer. In Kaggle Add-ons → Secrets, enable individual `HF_TOKEN`, `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY` entries, or one `FQ_SSIUU_DOTENV` Secret containing the same KEY=value format. `python-dotenv` parses it in memory without interpolation; no `.env` file is uploaded or written. The loader checks environment variables first and fetches only names required by the recipe. Never print a token, `os.environ`, or parsed dotenv values.

The default private source-bundle bootstrap requires no Git token. GitHub tokens are available to explicitly configured authenticated repository operations; they are never put in clone URLs. HF tokens serve gated input access and private artifact storage. Integration-only validation requires none.


In [ ]:
PARAMETERS = {'source_origin_sha': 'b884c5ef0a8bfe7812e583f88f73cb5c1eaf9a83', 'source_tree_sha': 'c7a75d5cdb048ade29c674b83541ad4b70ff823c', 'source_manifest_sha256': 'a8f5f147f986bfd136015d637ae1384e2524b3b9cdc8e57dd32268e452a56bd9', 'dataset_id': 'muhammadii/fq-ssiuu-c-source-b884c5e', 'orchestration_commit': 'b884c5ef0a8bfe7812e583f88f73cb5c1eaf9a83', 'recipe': 'configs/runs/work_unit_c_precision_v1.json'}


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import hashlib
import importlib.util
import json
from pathlib import Path
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone

if "STAGING_REQUIRED" in PARAMETERS.values():
    raise ValueError("Stage the notebook with immutable source identities first")
working = Path("/kaggle/working")
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:12]
checkout = working / ("checkout-" + run_id)
provenance_path = working / ("bootstrap-" + run_id + ".json")
output = working / "runs" / run_id
owned_checkout = False

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def git(*args):
    command = ["git", *args]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Source verification git command failed: " + result.stderr)
    return result.stdout.strip()

try:
    manifests = [p for p in Path("/kaggle/input").rglob("SOURCE_MANIFEST.json")
                 if sha256(p) == PARAMETERS["source_manifest_sha256"]]
    if len(manifests) != 1:
        raise ValueError("Exactly one attached source manifest must match its pinned hash")
    manifest_path = manifests[0]
    manifest = json.loads(manifest_path.read_text())
    if (manifest["source_origin_sha"] != PARAMETERS["source_origin_sha"]
            or manifest["source_tree_sha"] != PARAMETERS["source_tree_sha"]
            or manifest["bundle_filename"] != "fq-ssiuu-source.bundle"):
        raise ValueError("Source identity mismatch")
    bundle = manifest_path.parent / "fq-ssiuu-source.bundle"
    if sha256(bundle) != manifest["bundle_sha256"] or bundle.stat().st_size != manifest["bundle_size_bytes"]:
        raise ValueError("Source bundle hash or size mismatch")
    git("clone", "--no-checkout", "-b", "kaggle-snapshot", str(bundle), str(checkout))
    owned_checkout = True
    head = git("-C", str(checkout), "rev-parse", "HEAD")
    if (head != manifest["snapshot_commit_sha"]
            or git("-C", str(checkout), "rev-parse", "HEAD^{tree}") != PARAMETERS["source_tree_sha"]
            or git("-C", str(checkout), "rev-list", "--count", "--all") != "1"
            or len(git("-C", str(checkout), "rev-list", "--parents", "-n", "1", "HEAD").split()) != 1):
        raise ValueError("Source must have the exact tree and one parentless snapshot commit")
    git("-C", str(checkout), "checkout", "kaggle-snapshot")
    if git("-C", str(checkout), "status", "--porcelain"):
        raise ValueError("Source checkout is dirty")
    spec = importlib.util.spec_from_file_location("source_transport", checkout / "scripts/package_kaggle_source.py")
    transport = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(transport)
    transport.verify_bundle(manifest_path.parent, PARAMETERS["source_origin_sha"], PARAMETERS["source_manifest_sha256"])
    provenance = {**PARAMETERS, "snapshot_commit_sha": head, "source_manifest": manifest, "run_id": run_id}
    with provenance_path.open("x") as handle:
        json.dump(provenance, handle, indent=2)
    command = [sys.executable, str(checkout / "scripts/mother_runner.py"),
               "--recipe", str(checkout / PARAMETERS["recipe"]), "--output", str(output),
               "--provenance", str(provenance_path), "--run-id", run_id]
    result = subprocess.run(command, cwd=checkout)
    if result.returncode:
        raise RuntimeError("Recipe failed; raw run outputs and termination record are retained")
    print("Run complete:", run_id, "— preserve sealed outputs with both private archive receipts.")
except Exception as exc:
    with (working / ("bootstrap-failure-" + run_id + ".json")).open("x") as handle:
        json.dump({"run_id": run_id, "status": "FAIL", "error_type": type(exc).__name__,
                   "source_origin_sha": PARAMETERS["source_origin_sha"]}, handle, indent=2)
    raise
finally:
    if owned_checkout:
        shutil.rmtree(checkout)
